In [0]:
# ==============================================================================
# CÉLULA 1: Definição de Variáveis e Caminhos
# ==============================================================================

# Caminho completo do CSV no Volume do Databricks
csv_path = "/Volumes/workspace/default/unidades_consum/unidades_consumidoras.csv"

# Nome da tabela de destino no catálogo (salvando no schema default)
# Se você tiver criado um schema 'bronze', pode alterar para: "workspace.bronze.unidades_consumidoras"
target_table = "workspace.default.bronze_unidades_consumidoras"

print(f"Caminho do arquivo de origem: {csv_path}")
print(f"Tabela de destino Bronze: {target_table}")

Caminho do arquivo de origem: /Volumes/workspace/default/unidades_consum/unidades_consumidoras.csv
Tabela de destino Bronze: workspace.default.bronze_unidades_consumidoras


In [0]:
# ==============================================================================
# CÉLULA 2: Leitura Bruta 100% String
# ==============================================================================

# Ao não usar inferSchema, o Spark lê todas as colunas como StringType,
# preservando zeros à esquerda, caracteres especiais e formatos originais do CSV
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")  # Garante fidelidade total em string
    .load(csv_path)
)

# Todas as colunas aparecerão como string:
df_raw.printSchema()

root
 |-- id_uc: string (nullable = true)
 |-- cpf_cnpj: string (nullable = true)
 |-- nome_titular: string (nullable = true)
 |-- logradouro: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- cep: string (nullable = true)
 |-- classe: string (nullable = true)
 |-- subgrupo_tarifario: string (nullable = true)
 |-- alimentador: string (nullable = true)
 |-- data_ligacao: string (nullable = true)
 |-- status: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)



In [0]:
# ==============================================================================
# CÉLULA 3: Escrita na Bronze com Ingestion Datetime
# ==============================================================================
from pyspark.sql.functions import current_timestamp, col

(
    df_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .withColumn("source_file", col("_metadata.file_name"))
    .write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")  # Sobrescreve o schema anterior que tinha inferido inteiro
    .saveAsTable(target_table)
)

print(f"Tabela Bronze '{target_table}' recriada com todas as colunas em StringType!")

Tabela Bronze 'workspace.default.bronze_unidades_consumidoras' recriada com todas as colunas em StringType!


In [0]:
# ==============================================================================
# CÉLULA 4: Validação dos Registros Gravados
# ==============================================================================

# Leitura da tabela salva no metastore
df_bronze = spark.table(target_table)

# Visualização das 5 primeiras linhas
display(df_bronze.limit(5))

id_uc,cpf_cnpj,nome_titular,logradouro,numero,bairro,municipio,uf,cep,classe,subgrupo_tarifario,alimentador,data_ligacao,status,latitude,longitude,ingestion_datetime,source_file
UC100000,49281624044,Edvaldo Pereira Bezerra,Travessa Getulio Vargas,1575,Arruda,Recife,PE,55215987,Residencial,B1,ALM-REC-004,2011-12-20,ATIVA,-8.10,-34.89,2026-10-19T02:31:14.597Z,unidades_consumidoras.csv
UC100001,77922481000117,Farmacia Martins LTDA,Avenida dos Navegantes,1309,Rosarinho,Recife,Pernambuco,55760925,Comercial,B3,ALM-REC-035,2011-10-17,ATIVA,-8.06,-34.83,2026-10-19T02:31:14.597Z,unidades_consumidoras.csv
UC100002,84515901010,Leticia Oliveira,Avenida Padre Cicero,1301,Iputinga,Recife,PE,54496152,Residencial,B1,ALM-REC-024,1999-04-11,ATIVA,-8.00,-34.87,2026-10-19T02:31:14.597Z,unidades_consumidoras.csv
UC100003,49967822341,Lucas Monteiro Amorim,Alameda Domingos Ferreira,129,Jardim Caetés,Abreu e Lima,PE,50259494,Residencial,B1,ALM-ABR-001,2019-11-25,ATIVA,-7.91,-34.89,2026-10-19T02:31:14.597Z,unidades_consumidoras.csv
UC100004,15474740174,Camila Souza Gomes,Avenida Santo Antonio,1207,Iputinga,Recife,PE,56760779,Residencial,B1,ALM-REC-024,2002-01-08,ATIVA,-8.04,-34.90,2026-10-19T02:31:14.597Z,unidades_consumidoras.csv


In [0]:
# ==============================================================================
# Verificação de Quantidade de Registros do CSV
# ==============================================================================

# Caminho do arquivo na Landing Zone
csv_path = "/Volumes/workspace/default/unidades_consum/unidades_consumidoras.csv"

# Leitura direta do arquivo CSV
df_check = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(csv_path)
)

# Contagem total de linhas de dados (excluindo a linha de cabeçalho)
total_registros = df_check.count()

print(f"Total de registros no CSV: {total_registros:,}".replace(",", "."))

Total de registros no CSV: 46.461
